# Crop Disease Detection
### MobileNetV2 Feature Extraction + XGBoost Classification

**Pipeline overview:**
1. Scan PlantVillage dataset → stratified train/val/test split
2. Extract 1,280-dim features using frozen MobileNetV2 + GlobalAveragePooling2D
3. Apply SMOTE oversampling + inverse-frequency sample weights
4. Train XGBoost with Optuna hyperparameter search
5. Evaluate on held-out test set + generate plots
6. Single-image inference demo

## 0. Setup

In [1]:
import sys, os
from pathlib import Path

# Add project root to path so all src.* imports work
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import tensorflow as tf
import xgboost as xgb
import sklearn

import config
from src.utils import setup_output_dirs, get_logger

setup_output_dirs()
logger = get_logger("notebook")

print(f"TensorFlow {tf.__version__}")
print(f"XGBoost  {xgb.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print("All imports OK")

TensorFlow 2.16.1
XGBoost  2.0.3
scikit-learn 1.4.1.post1
All imports OK


## 1. Data Pipeline
Scan PlantVillage folder → build class index → stratified 70/15/15 split.

In [2]:
from src.data_pipeline import build_class_index, scan_dataset, stratified_split

print(f"Dataset path: {config.DATA_DIR}")

# Build sorted class → integer index from folder names
class_index = build_class_index(config.DATA_DIR)

print(f"\nClasses found ({len(class_index)}):")
for name, idx in class_index.items():
    folder = config.DATA_DIR / name
    count = len(list(folder.glob("*.jpg"))) + len(list(folder.glob("*.JPG")))
    print(f"  {idx:2d}  {name:<45} ({count:4d} images)")

# Full dataframe: filepath, label_str, label_int
df = scan_dataset(config.DATA_DIR, class_index)
counts = df.groupby('label_int').size()
print(f"\nTotal images: {len(df):,}")
print(f"Imbalance ratio: {counts.max()/counts.min():.1f}x  "
      f"(min={counts.min()} {df.groupby('label_int')['label_str'].first()[counts.idxmin()]}, "
      f"max={counts.max()} {df.groupby('label_int')['label_str'].first()[counts.idxmax()]})")

Dataset path: /Users/jahnavisrilekhamolleti/Documents/PersonalProjects/CapstoneProjects/crop_disease_classifier/data/raw/PlantVillage

Classes found (15):
  0  Pepper__bell___Bacterial_spot       (1000 images)
  1  Pepper__bell___healthy              (1478 images)
  2  Potato___Early_blight               (1000 images)
  3  Potato___Late_blight                (1000 images)
  4  Potato___healthy                    ( 152 images)
  5  Tomato_Bacterial_spot               (2127 images)
  6  Tomato_Early_blight                 (1000 images)
  7  Tomato_Late_blight                  (1909 images)
  8  Tomato_Leaf_Mold                    ( 952 images)
  9  Tomato_Septoria_leaf_spot           (1771 images)
 10  Tomato_Spider_mites_Two_spotted_spider_mite (1676 images)
 11  Tomato__Target_Spot                 (1404 images)
 12  Tomato__Tomato_YellowLeaf__Curl_Virus (3208 images)
 13  Tomato__Tomato_mosaic_virus         ( 373 images)
 14  Tomato_healthy                      (1591 images)

Total ima

In [3]:
# Stratified split — same class proportions in all three splits
train_df, val_df, test_df = stratified_split(df)

print("Split sizes (stratified 70/15/15):")
print(f"  Train : {len(train_df):6,} images")
print(f"  Val   : {len(val_df):6,} images")
print(f"  Test  : {len(test_df):6,} images")

all_present = (
    train_df['label_int'].nunique() == len(class_index) and
    val_df['label_int'].nunique()   == len(class_index) and
    test_df['label_int'].nunique()  == len(class_index)
)
print(f"\nAll {len(class_index)} classes present in every split: {all_present}")

Split sizes (stratified 70/15/15):
  Train : 14,446 images
  Val   :  3,104 images
  Test  :  3,097 images

All 15 classes present in every split: True


## 2. Feature Extraction — Frozen MobileNetV2

MobileNetV2 (pre-trained ImageNet weights, all layers **frozen**) + GlobalAveragePooling2D  
Input: `(224, 224, 3)` → Output: `(1280,)` feature vector per image.

Features are cached to `.npz` files — extraction runs **once** (~15–30 min on CPU);  
all subsequent training runs load from cache in seconds.

In [4]:
from src.feature_extractor import build_feature_model

# Build frozen feature model — downloads ImageNet weights on first run
feature_model = build_feature_model()

print("MobileNetV2 feature extractor")
print(f"Input  shape : {feature_model.input_shape}")
print(f"Output shape : {feature_model.output_shape}")
print(f"Total params      : {feature_model.count_params():,}")
print(f"Trainable params  : {sum(tf.size(v).numpy() for v in feature_model.trainable_variables):<10} ← all layers frozen")
print(f"Non-trainable     : {sum(tf.size(v).numpy() for v in feature_model.non_trainable_variables):,}")

MobileNetV2 feature extractor
Input  shape : (None, 224, 224, 3)
Output shape : (None, 1280)
Total params      : 2,257,984
Trainable params  : 0          ← all layers frozen
Non-trainable     : 2,257,984


In [5]:
from src.feature_extractor import features_cached, load_features, save_features, extract_features
from src.data_pipeline import build_augmentation_pipeline, build_preprocessing_pipeline, image_generator, num_batches
from src.utils import build_label_encoder, save_label_encoder, load_label_encoder

all_cached = all(
    features_cached(p)
    for p in [config.TRAIN_FEATURES_PATH, config.VAL_FEATURES_PATH, config.TEST_FEATURES_PATH]
)

if all_cached:
    print("Feature caches already exist — loading from disk.")
    print(f"  {config.TRAIN_FEATURES_PATH.name}")
    print(f"  {config.VAL_FEATURES_PATH.name}")
    print(f"  {config.TEST_FEATURES_PATH.name}")
    X_train, y_train = load_features(config.TRAIN_FEATURES_PATH)
    X_val,   y_val   = load_features(config.VAL_FEATURES_PATH)
    X_test,  y_test  = load_features(config.TEST_FEATURES_PATH)
else:
    # One-time extraction (~15-30 min on CPU)
    print("Extracting features — this runs once, results cached to .npz")
    aug_fn = build_augmentation_pipeline()     # augment + preprocess_input (train)
    pre_fn = build_preprocessing_pipeline()    # preprocess_input only (val/test)

    train_gen = image_generator(train_df["filepath"].tolist(), train_df["label_int"].values,
                                batch_size=config.BATCH_SIZE, preprocess_fn=aug_fn)
    X_train, y_train = extract_features(feature_model, train_gen, num_batches(len(train_df)), desc="Train")
    save_features(X_train, y_train, config.TRAIN_FEATURES_PATH)

    val_gen = image_generator(val_df["filepath"].tolist(), val_df["label_int"].values,
                              batch_size=config.BATCH_SIZE, preprocess_fn=pre_fn)
    X_val, y_val = extract_features(feature_model, val_gen, num_batches(len(val_df)), desc="Val")
    save_features(X_val, y_val, config.VAL_FEATURES_PATH)

    test_gen = image_generator(test_df["filepath"].tolist(), test_df["label_int"].values,
                               batch_size=config.BATCH_SIZE, preprocess_fn=pre_fn)
    X_test, y_test = extract_features(feature_model, test_gen, num_batches(len(test_df)), desc="Test")
    save_features(X_test, y_test, config.TEST_FEATURES_PATH)

    # Save label encoder
    label_names = sorted(class_index.keys())
    le = build_label_encoder(label_names)
    save_label_encoder(le, config.LABEL_ENCODER_PATH)

print(f"\nLoaded shapes:")
print(f"  X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"  X_val  : {X_val.shape}   y_val  : {y_val.shape}")
print(f"  X_test : {X_test.shape}   y_test : {y_test.shape}")

# Load label encoder
le = load_label_encoder(config.LABEL_ENCODER_PATH)
label_names = list(le.classes_)

Feature caches already exist — loading from disk.
  train_features_224.npz
  val_features_224.npz
  test_features_224.npz

Loaded shapes:
  X_train: (14446, 1280)  y_train: (14446,)
  X_val  : (3104, 1280)   y_val  : (3104,)
  X_test : (3097, 1280)   y_test : (3097,)


## 3. Class Imbalance Visualisation

In [6]:
unique, counts = np.unique(y_train, return_counts=True)
short_names = [n.split("___")[-1].replace("_", " ") if "___" in n else n.split("__")[-1].replace("_", " ")
               for n in label_names]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e74c3c' if c < 300 else '#f39c12' if c < 1000 else '#2ecc71' for c in counts]
bars = ax.bar(range(len(unique)), counts, color=colors, edgecolor='white')
ax.set_xticks(range(len(unique)))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Training samples")
ax.set_title("Class Distribution in Training Set (before SMOTE)\n"
             "Red = minority (<300), Orange = moderate (<1000), Green = majority")
ax.axhline(y=1000, color='gray', linestyle='--', alpha=0.5, label='SMOTE target')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Imbalance ratio: {counts.max()}/{counts.min()} = {counts.max()/counts.min():.1f}x")

## 4. Preprocessing — SMOTE Oversampling

SMOTE synthesises new minority-class samples by interpolating between real samples in the 1,280-dim feature space.  
Applied **only to training features** — val and test are never oversampled.

In [7]:
from src.preprocessor import apply_smote, run_preprocessing

# Show distribution before SMOTE
print("Class distribution BEFORE SMOTE:")
u, c = np.unique(y_train, return_counts=True)
for cls, cnt in zip(u, c):
    marker = "  ← minority" if cnt < 300 else ""
    print(f"  class {cls:2d}: {label_names[cls]:<45} {cnt:4d} samples{marker}")

print(f"\nRunning SMOTE (target={config.SMOTE_TARGET_PER_CLASS} per class)...")

# Apply SMOTE — only on training data
X_train_bal, y_train_bal = apply_smote(
    X_train, y_train,
    target_per_class=config.SMOTE_TARGET_PER_CLASS,
    k_neighbors=config.SMOTE_K_NEIGHBORS,
)

print(f"\nClass distribution AFTER SMOTE:")
u2, c2 = np.unique(y_train_bal, return_counts=True)
print(f"  All {len(u2)} classes → {c2.min()} samples each")
print(f"  Total training samples: {len(y_train):,} → {len(y_train_bal):,}  "
      f"(+{len(y_train_bal)-len(y_train):,} synthetic samples)")

Class distribution BEFORE SMOTE:
  class  0: Pepper__bell___Bacterial_spot        700 samples
  class  1: Pepper__bell___healthy              1034 samples
  class  2: Potato___Early_blight                700 samples
  class  3: Potato___Late_blight                 700 samples
  class  4: Potato___healthy                     106 samples  ← minority
  class  5: Tomato_Bacterial_spot               1489 samples
  class  6: Tomato_Early_blight                  700 samples
  class  7: Tomato_Late_blight                  1336 samples
  class  8: Tomato_Leaf_Mold                     666 samples
  class  9: Tomato_Septoria_leaf_spot           1239 samples
  class 10: Tomato_Spider_mites                 1173 samples
  class 11: Tomato__Target_Spot                  982 samples
  class 12: Tomato__Tomato_YellowLeaf__Curl_Virus 2245 samples
  class 13: Tomato__Tomato_mosaic_virus           261 samples
  class 14: Tomato_healthy                       1113 samples

Running SMOTE (target=2000 per clas

## 5. XGBoost Training

Two-phase training:
1. **Baseline** — fixed hyperparameters from `config.XGB_BASE_PARAMS`
2. **Optuna tuning** — TPE sampler searches 20 trials × 3-fold CV → best params
3. **Retrain** — final model trained on full training set with Optuna best params

In [8]:
from src.classifier import train_baseline, compute_sample_weights

# Compute inverse-frequency sample weights so rare classes get higher gradient signal
sample_weights = compute_sample_weights(y_train_bal)
print(f"Sample weights applied:")
print(f"  min weight: {sample_weights.min():.3f}  max weight: {sample_weights.max():.3f}  "
      f"ratio: {sample_weights.max()/sample_weights.min():.1f}x")
print(f"  (inverse-frequency weighting; SMOTE balancing makes ratio near 1.0)")

print(f"\nTraining baseline XGBoost model...")
baseline_model = train_baseline(X_train_bal, y_train_bal, X_val, y_val)

Sample weights applied:
  min weight: 0.898  max weight: 1.008  ratio: 1.1x
  (inverse-frequency weighting; SMOTE balancing makes ratio near 1.0)

Training baseline XGBoost model...
[0]    validation_0-mlogloss: 2.57231
[50]   validation_0-mlogloss: 0.98432
[100]  validation_0-mlogloss: 0.75614
[150]  validation_0-mlogloss: 0.65821
[200]  validation_0-mlogloss: 0.60134
[250]  validation_0-mlogloss: 0.57023
[300]  validation_0-mlogloss: 0.54891
[350]  validation_0-mlogloss: 0.53402
[400]  validation_0-mlogloss: 0.52817
[411]  validation_0-mlogloss: 0.52743

Baseline val macro F1: 0.5746


In [9]:
from src.classifier import run_tuning

# Optuna hyperparameter search: 20 trials × 3-fold stratified CV
# TPE sampler + MedianPruner (drops unpromising trials early)
study = run_tuning(X_train_bal, y_train_bal, n_trials=config.OPTUNA_N_TRIALS)

Optuna search: 20 trials × 3-fold CV = 60 XGBoost runs
Hyperparameter search space:
  n_estimators         [200, 600]
  learning_rate        [0.01, 0.3]
  max_depth            [3, 6]
  subsample            [0.6, 1.0]
  colsample_bytree     [0.3, 0.8]
  min_child_weight     [1, 10]
  reg_lambda           [0.5, 10.0]
  reg_alpha            [0.0, 2.0]
Trial #0  starting | n_est=312  lr=0.0213  depth=5  subsample=0.71  col_bt=0.62  mcw=3  λ=2.31  α=0.45
Trial #0  result   | macro F1 = 0.8821 ± 0.0031  (88.21%)
Trial #1  starting | n_est=487  lr=0.1124  depth=3  subsample=0.93  col_bt=0.41  mcw=7  λ=6.12  α=1.23
Trial #1  result   | macro F1 = 0.9102 ± 0.0018  (91.02%)
Trial #2  starting | n_est=521  lr=0.0891  depth=4  subsample=0.81  col_bt=0.55  mcw=2  λ=3.87  α=0.91
Trial #2  result   | macro F1 = 0.9287 ± 0.0022  (92.87%)
Trial #3  result   | macro F1 = 0.9014 ± 0.0041  (90.14%)
Trial #4  result   | macro F1 = 0.9198 ± 0.0029  (91.98%)
Trial #5  result   | macro F1 = 0.9351 ± 0.0017  (

In [10]:
from src.classifier import retrain_best, save_model

print("Retraining with Optuna best params on full training set...")
final_model = retrain_best(
    X_train_bal, y_train_bal,
    X_val, y_val,
    study.best_params,
)

save_model(final_model, config.XGB_MODEL_PATH)
print(f"Model saved to {config.XGB_MODEL_PATH.relative_to(PROJECT_ROOT)}")

Retraining with Optuna best params on full training set...
[0]    validation_0-mlogloss: 2.38124
[50]   validation_0-mlogloss: 0.72341
[100]  validation_0-mlogloss: 0.58912
[150]  validation_0-mlogloss: 0.52341
[200]  validation_0-mlogloss: 0.48891
[250]  validation_0-mlogloss: 0.46234
[300]  validation_0-mlogloss: 0.44891
[350]  validation_0-mlogloss: 0.43712
[400]  validation_0-mlogloss: 0.43102
[438]  validation_0-mlogloss: 0.42891

Tuned model val macro F1: 0.7341
Model saved to outputs/models/xgb_model.json


## 6. Optuna Trial History

In [11]:
from src.evaluate import plot_optuna_history

# Plot macro F1 across all Optuna trials — shows search convergence
plot_optuna_history(study, config.PLOTS_DIR / "optuna_history.png")
print(f"Optuna history plot saved to outputs/plots/optuna_history.png")

# Display inline
img = plt.imread(str(config.PLOTS_DIR / "optuna_history.png"))
plt.figure(figsize=(10, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

Optuna history plot saved to outputs/plots/optuna_history.png


## 7. Test Set Evaluation
> **Grading cell** — evaluates the final model on the held-out test set (3,097 images, never seen during training or hyperparameter search).

In [12]:
from src.evaluate import compute_all_metrics, save_classification_report

# Evaluate on test set — this cell runs the final assessment
metrics = compute_all_metrics(final_model, X_test, y_test, label_names)

print("=" * 50)
print("TEST SET RESULTS  (3,097 held-out images)")
print("=" * 50)
print(f"  Accuracy      : {metrics['accuracy']*100:.2f}%")
print(f"  Macro F1      : {metrics['macro_f1']:.4f}")
print(f"  Weighted F1   : {metrics['weighted_f1']:.4f}")
print(f"  Top-3 Accuracy: {metrics['top3_accuracy']*100:.2f}%")
print("=" * 50)

# Per-class breakdown
print("\nPer-class breakdown:")
print(f"  {'':44} {'precision':>9}  {'recall':>6}  {'f1':>5}   {'support':>7}")
for i, name in enumerate(label_names):
    p = metrics['per_class'][i]['precision']
    r = metrics['per_class'][i]['recall']
    f = metrics['per_class'][i]['f1']
    s = metrics['per_class'][i]['support']
    print(f"  {name:<46} {p:>8.2f}  {r:>6.2f}  {f:>5.2f}  {s:>7}")
print(f"  {'─'*73}")
print(f"  {'macro avg':<46} {metrics['avg_precision']:>8.2f}  {metrics['avg_recall']:>6.2f}  {metrics['macro_f1']:>5.2f}  {len(y_test):>7}")
print(f"  {'weighted avg':<46} {metrics['weighted_precision']:>8.2f}  {metrics['weighted_recall']:>6.2f}  {metrics['weighted_f1']:>5.2f}  {len(y_test):>7}")

# Save classification report to disk
save_classification_report(final_model, X_test, y_test, label_names,
                           config.REPORTS_DIR / "classification_report.txt")

TEST SET RESULTS  (3,097 held-out images)
  Accuracy      : 73.81%
  Macro F1      : 0.7125
  Weighted F1   : 0.7241
  Top-3 Accuracy: 93.19%

Per-class breakdown:
                                             precision  recall    f1   support
  Pepper__bell___Bacterial_spot                 0.85    0.87   0.86      149
  Pepper__bell___healthy                        0.94    0.98   0.96      221
  Potato___Early_blight                         0.82    1.00   0.90      150
  Potato___Late_blight                          0.97    0.67   0.79      150
  Potato___healthy                              0.89    0.74   0.81       23
  Tomato_Bacterial_spot                         0.95    0.22   0.36      319
  Tomato_Early_blight                           0.86    0.16   0.27      150
  Tomato_Late_blight                            0.85    0.84   0.84      287
  Tomato_Leaf_Mold                              0.80    0.72   0.76      143
  Tomato_Septoria_leaf_spot                     0.43    0.96   0

## 8. Confusion Matrix

In [13]:
from src.evaluate import plot_confusion_matrix

y_pred = final_model.predict(X_test)

# 15×15 confusion matrix — rows=true, cols=predicted
plot_confusion_matrix(y_test, y_pred, label_names,
                      config.PLOTS_DIR / "confusion_matrix.png")

img = plt.imread(str(config.PLOTS_DIR / "confusion_matrix.png"))
plt.figure(figsize=(12, 10))
plt.imshow(img)
plt.axis('off')
plt.title("Confusion Matrix — Test Set", pad=10)
plt.tight_layout()
plt.show()

## 9. Per-Class F1 Chart

In [14]:
from src.evaluate import plot_per_class_f1

# Horizontal bar chart — sorted ascending so worst classes appear at the top
plot_per_class_f1(metrics["per_class"], label_names,
                  config.PLOTS_DIR / "per_class_f1.png")

img = plt.imread(str(config.PLOTS_DIR / "per_class_f1.png"))
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 10. XGBoost mlogloss Training Curve
Shows how validation loss decreases as more trees are added — equivalent to loss vs. epochs for neural networks.

In [15]:
# Extract evaluation log (mlogloss per estimator on validation set)
evals_result = final_model.evals_result()

val_loss = evals_result.get("validation_0", {}).get("mlogloss", [])

if val_loss:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(val_loss, color='#e74c3c', linewidth=1.5, label='Validation mlogloss')
    ax.set_xlabel("Number of Trees (estimators)")
    ax.set_ylabel("mlogloss")
    ax.set_title("XGBoost Validation Loss vs Number of Trees\n"
                 "(lower = better; equivalent to loss-vs-epochs for neural networks)")
    ax.legend()
    ax.grid(alpha=0.3)
    # Annotate minimum
    min_idx = int(np.argmin(val_loss))
    ax.axvline(x=min_idx, color='gray', linestyle='--', alpha=0.7)
    ax.annotate(f'Best: {val_loss[min_idx]:.4f}\nat tree {min_idx}',
                xy=(min_idx, val_loss[min_idx]),
                xytext=(min_idx + 20, val_loss[min_idx] + 0.05),
                arrowprops=dict(arrowstyle='->', color='black'))
    plt.tight_layout()
    plt.savefig(config.PLOTS_DIR / "xgb_loss_curve.png", dpi=150)
    plt.show()
else:
    print("No evaluation log found — model was loaded from disk.")

## 11. Single-Image Inference Demo
End-to-end prediction pipeline for a field photo — as a farmer would use it.

In [16]:
from src.inference import load_inference_pipeline, predict_disease, format_prediction_report

# Load the full inference pipeline (feature model + xgb model + label encoder)
pipeline = load_inference_pipeline(config.MODELS_DIR)

# Pick a random test image for demo
sample_idx = np.random.default_rng(42).integers(0, len(test_df))
sample_path = test_df.iloc[sample_idx]["filepath"]
true_label  = test_df.iloc[sample_idx]["label_str"]

print("Running inference on a sample test image...\n")
predictions = predict_disease(sample_path, pipeline, top_k=3)
print(format_prediction_report(predictions))

top1_correct = predictions[0]['raw_label'] == true_label
print(f"\nTrue label: {true_label}  {'✅ Correct (top-1)' if top1_correct else '❌ Wrong'}")

Running inference on a sample test image...

Crop Disease Classification Result
  #1  Tomato — Late blight              84.3%
  #2  Tomato — Early blight              9.1%
  #3  Tomato — Bacterial spot            4.2%

True label: Tomato_Late_blight  ✅ Correct (top-1)


In [17]:
# Display the sample image alongside the top-3 predictions
from PIL import Image

img = Image.open(str(sample_path)).convert("RGB")

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 5))

ax_img.imshow(img)
ax_img.set_title(f"Input image\nTrue: {true_label}", fontsize=9)
ax_img.axis('off')

# Horizontal confidence bars
labels = [f"{p['crop']} — {p['disease']}" for p in predictions]
confs  = [p['confidence'] * 100 for p in predictions]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars   = ax_bar.barh(range(len(predictions)), confs, color=colors)
ax_bar.set_yticks(range(len(predictions)))
ax_bar.set_yticklabels([f"#{p['rank']} {l}" for p, l in zip(predictions, labels)], fontsize=8)
ax_bar.set_xlabel("Confidence (%)")
ax_bar.set_title("Top-3 Predictions")
ax_bar.set_xlim(0, 100)
for bar, conf in zip(bars, confs):
    ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"{conf:.1f}%", va='center', fontsize=9)
ax_bar.invert_yaxis()

plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Test Accuracy | 73.81% |
| Macro F1 | 0.7125 |
| Weighted F1 | 0.7241 |
| **Top-3 Accuracy** | **93.19%** |
| Best Optuna CV F1 | 0.9462 (Trial #6) |

**Key findings:**
- Top-3 accuracy of 93.19% confirms the model correctly places the true disease in the top 3 predictions for 9 in 10 images
- Visually distinct diseases (Pepper healthy, Potato Early blight, Tomato TYLCV) achieve F1 ≥ 0.90
- Visually similar Tomato diseases (Early blight, Bacterial spot, Septoria) remain challenging for a frozen feature extractor; fine-tuning MobileNetV2 top layers is the recommended next step